# Introduction to Working with Data

Data analysis rarely begins with a clean dataset. This notebook covers the work that comes first: getting data into Python, understanding what is actually in it, and reshaping it into something we can analyze. We start with the vocabulary we use to describe datasets, then work through reshaping, cleaning, combining, and sampling.

Before starting, work through the [Statistical Concepts](../../03_data_analytics_statistics/statistical_concepts.ipynb) notebook. This module assumes you are comfortable importing packages, loading a dataset, and running cells.

We will work with two small datasets in this folder: a set of invoices and a set of shipping records. They are deliberately tiny so that you can read every row and see exactly what each step does. The [diamonds exercise](../04.2_diamonds_exercise) and the [sales exercise](../04.3_sales_exercise) then apply these same tools to much larger data.

Before we get started, some important things to remember: 

* *Make sure you understand the data you are working with.* Some things to ask yourself: How was the data gathered or created? Where did the data come from? How is the data structured? What do the variables represent?

* *Validate the data before you analyze it.* Be an auditor of the data you are working with. Check a sample of the data for accuracy. Review descriptives (e.g., means, medians, standard deviations, and distributional characteristics) for reasonableness.

 * *"Avoid embarrassment by being your own best skeptic..."* ([Mostly Harmless Econometrics](https://www.amazon.com/Mostly-Harmless-Econometrics-Empiricists-Companion/dp/0691120358)). Be clear about limitations of the data and your analyses...avoid overselling!

## General housekeeping items
As in the previous notebooks, select your kernel before running anything. Click **Select Kernel** in the top right of this notebook, choose **Python Environments...**, and select the `.venv` environment you created in [Getting Started](../../01_getting_started/course_intro.ipynb).

We will use `pandas` extensively in this course for data wrangling, cleaning, and analysis. See the [documentation](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html) for an extensive list of functions and methods. `pandas` is a Python library for data analysis and manipulation, built on top of `numpy`. It is especially good with tabular data, meaning anything that looks like a spreadsheet.

In [ ]:
import numpy as np
import pandas as pd

## Structured and unstructured data

**Structured data** is a generic term for data that is organized in some fashion. Structured data is relatively easy to work with. Corporate databases, a grade book, your calendar, and investment returns are all examples.

**Unstructured** (or relatively unstructured) **data** is not organized or well curated for analysis. Text, videos, and photos are common examples. Most "machine-readable" data has some structure but still needs to be cleaned and prepared before it can be analyzed.

In practice, the line between structured and unstructured data is blurry, and it is more useful to think of structure as a **continuum**. At one end sits a well-designed "tidy" database table. Moving down the continuum, a spreadsheet may contain merged cells, stray notes, and inconsistent formatting. A PDF financial statement has a visual structure that a human reads easily but a computer does not. Further along, an email or an earnings call transcript has some regularity (a sender, a date, sections of text) but no columns at all. At the far end are photos and video, which are not readily organized for analysis.

## Tidy data

"Tidy" data is structured data that is ready to analyze. In a tidy dataset, columns are variables (e.g., net income, assets, revenues) and rows are observations or units (e.g., companies or company-years). The diamonds dataset we worked with in [Statistical Concepts](../../03_data_analytics_statistics/statistical_concepts.ipynb) is a good example of a tidy dataset. Note that structured data is not necessarily tidy, and getting from one to the other is most of the job: data "tidying" or "wrangling" often takes up most of the time spent on a data analysis project. For more on this idea, refer to [Hadley Wickham's tidy data guide](https://vita.had.co.nz/papers/tidy-data.pdf). 

## Common data forms

From an analysis perspective, data generally take one of three forms, depending on how many subjects they cover and whether they cover them over time.

| Form | What it covers | Example |
|------|----------------|---------|
| Cross-sectional | Many subjects (people, companies, countries, etc.) at a point in time | Financial data for all public companies in 2019 |
| Time-series | One subject over time | Tesla's financial data from 2010-2019 |
| Panel | Many subjects over time | Financial data for all public companies from 2010-2019 |

### In-class discussion: the Coozie Data
What form was the Coozie Data in [Python Basics](../../02_python_basics/python_basics.ipynb)? Was the original Coozie Data in tidy format?

## Variables and common variable types

As noted earlier, "tidy" data has observations in rows and variables in columns. Variables contain features or characteristics of each observation, and can take different (or "varying") values. Variables come in a handful of types:


| Type | Description | Examples | Common `pandas` dtype |
|------|-------------|----------|-----------------------|
| Factor or categorical | Take a limited number of values | *Ordinal:* education level, grades, bond ratings<br>*Nominal:* geographic location, industry classification, colors | `category`, or `object` if left as raw text |
| Discrete numeric | Take specific values like whole numbers or integers | Inventory counts, population | `int64` |
| Indicator, Boolean, and logical | Take binary values | True/false, win/lose, heads/tails, 1/0 | `bool` |
| Continuous numeric | Take (at least in theory) an infinite number of values including fractions | Earnings per share, distance, time | `float64` |
| Character | Capture text or "strings" | Names, phrases, sentences | `object` (or `string`) |
| Date | Capture dates and times | "09/02/2020", "September 2nd, 2020", "Sept-2-2020", "8:00AM" | `datetime64[ns]` |

A few things to watch for. `object` is the dtype `pandas` falls back on whenever a column holds text or a mix of types, so seeing `object` where you expected a number is a reliable sign that something needs cleaning. Also, a date that `pandas` has not been told is a date is just text, which means it will sort as text rather than by date... which may not be chronological.

## Conventions for object and variable names

Whenever possible, avoid using the "spacebar" when naming objects, datasets, and variables. Two common naming conventions are `snake_case`, where all characters are lower case and underscores represent spaces (`diamonds_data`), and `camelCase`, where characters are lower case except the first letter of a new word (`diamondsData`). Snake case is my preference, and it is what you will see throughout this course.

## Importing data

We can import (and export) many file and data types into Python using `pandas` and other packages: comma delimited files (csv), tab delimited files (tsv), Excel files (xls), HTML files, JSON, and many more. In some cases, we may need other packages besides `pandas` to import certain file types or work with very large datasets efficiently. If you run into something new... chances are... there's a package for that!

Let's bring in the two datasets for this lesson. First, a small set of invoices. Look at it closely and see if any issues stand out. 

In [ ]:
invoices = pd.read_csv('invoice_data.csv')

invoices

In [ ]:
invoices.info()

Five rows, six columns, and already a few problems: one invoice appears twice (1002) and `invoice_datetime` is an 'object' rather than a date. Harder to see is that one customer name has a stray trailing space (Madison C.). These are common data issues, and we will fix each of these below.

Now, let's explore the shipping records that go with those invoices.

In [ ]:
shipping_data = pd.read_csv('shipping_data.csv')

shipping_data

In [ ]:
shipping_data.info()

This seems like useful data, but it isn't organized very well. Notice we have several data types and features all in the same column. Below, we will work to organize this data into more workable form.

## Preparing messy data for analysis

Unfortunately, we do not always get our data in "tidy" format, and even well-structured data requires some preparation before analysis. Cleaning and preparing data is not the most appealing task in data analysis. However, the ability to convert unstructured or messy data to tidy data is what distinguishes a good data scientist. That is, the better you can effectively extract, clean, and prepare data, the wider range of data you can work with (and the more valuable your skills are!).

## Organizing and reshaping data

Notice the shape of the shipping data above. Each invoice takes up four rows, one per attribute, with every value crammed into a single `value` column. The data is not in a shape that we can work with. There is no `weight` column to clean, no date column to parse, and individual invoices represent multiple rows. **Reshaping therefore comes first**: putting one observation in each row and each variable in its own column makes it much easier to clean individual variables and combine this data with other datasets.

The `pandas` library helps us organize data structures like these. `melt()` and `pivot()` are used to reshape datasets, converting rows into columns and vice versa. Other useful tools such as `str.cat()` and `str.split()` allow us to combine and separate text values.

To go from long to wide, use `pivot()`. The general form is `pivot(index, columns, values)`, where `index` is the column that identifies an observation, `columns` holds the names of the new variables, and `values` holds the data that fills them.

In [ ]:
shipping_data_tidy = shipping_data.pivot(
    index='invoice',
    columns='attribute',
    values='value'
).reset_index() # Move invoice out of the index and back into a column

shipping_data_tidy

Twelve rows became three, and each attribute now has its own column. This is a dataset we can work with. Note that we used `reset_index` here. The purpose is to reset the row index and move the invoice numbers back into a column. More on this function below.

Reshaping runs in both directions. `melt()` is the reverse of `pivot()`, taking wide data back to long. The general form is `melt(id_vars, var_name, value_name)`, where `id_vars` identifies the columns that stay fixed for each observation, `var_name` gives the name of the new column that holds the former variable names, and `value_name` gives the name of the new column that holds their values. Let's confirm we can recover the original data in long form.

In [ ]:
shipping_data_tidy.melt(
    id_vars='invoice',
    var_name='attribute',
    value_name='value'
)

Now that both the invoices and shipping datasets are in a workable shape, we will work with the shipping data first (dates, number formats, and a missing value), then the invoices (text and a duplicate row).

### Working with dates and times

Dates are notoriously hard to work with. Fortunately, the `pandas` library makes this process (relatively) easy for most date and time formats. When parsing dates and times, a 4-digit year is `%Y`, month is `%m`, and day is `%d`, so something like `pd.to_datetime('20101006', format='%Y%m%d')` would transform October 6, 2010 expressed as "20101006" into a proper date. Note here, that it is important to get the date formatted accurately. For example, 20101006 could mean October 6, 2010 or it could mean June 10, 2010, depending on the date format.

The syntax is flexible to other kinds of date formats. For example: if the date is formatted as "October 6, 2010" we could use `pd.to_datetime('October 6, 2010', format='%B %d, %Y')`. We don't cover all of the different dates and formats here, however, most formats are convertible into a workable date-time. Claude and Copilot tools are useful in working through this. Remember to be careful and check your work! Even widely used tools like Excel have introduced serious date-related errors, as reported in [An alarming number of scientific papers contain Excel errors - The Washington Post](https://www.washingtonpost.com/news/wonk/wp/2016/08/26/an-alarming-number-of-scientific-papers-contain-excel-errors/).

Let's work with dates in our example datasets. Start by looking at what we are dealing with.

In [ ]:
shipping_data_tidy

Notice that the dates are not properly formatted. Let's convert to date format.

In [ ]:
shipping_data_tidy['shipping_datetime'] = pd.to_datetime(
    shipping_data_tidy['shipping_datetime'],
    format='%b-%d-%Y %I%p' # e.g., Feb-7-2026 8am
)

shipping_data_tidy['shipping_receiving_datetime'] = pd.to_datetime(
    shipping_data_tidy['shipping_receiving_datetime'],
    format='%m/%d/%y %I%p' # e.g., 2/10/26 10am
)

shipping_data_tidy

Compare the two date columns to the cell above. Both now read as full timestamps like `2026-02-07 08:00:00`, which means `pandas` is storing them as real dates rather than text. Now we can sort by them, subtract them from each other, or pull the month out of them.

Now let's work with the dates in the invoices dataset.

In [ ]:
invoices

In [ ]:
invoices['invoice_datetime'] = pd.to_datetime(
    invoices['invoice_datetime'],
    format='%Y-%m-%d %H:%M:%S'
)

invoices

### Number formats

Numbers in Python are stored and displayed in different formats, and we will generally need to convert numbers stored as "text" or "string" to numeric before we can compute anything with them.

The dates are parsed, but the columns created from the `value` column are still stored as text.

In [ ]:
shipping_data_tidy

Notice that `weight` comes back as `object`, which is how `pandas` reports text. `pd.to_numeric()` converts it.

In [ ]:
shipping_data_tidy['weight'] = pd.to_numeric(shipping_data_tidy['weight'])

shipping_data_tidy

#### `int64` versus `float64`

Notice that the numeric columns did not all land on the same type. `pandas` distinguishes two kinds of number: an `int64` holds whole numbers and stores them exactly, while a `float64` holds numbers with decimals and stores them only *approximately*. The shipping data now has one of each, `invoice` as `int64` and `weight` as `float64`, and the invoice data does too: `quantity` is an `int64` because you cannot ship 2.5 units, and `amount` is a `float64`.

That approximation is worth seeing, because it catches people off guard. A `float64` stores its value in binary, and many decimal fractions have no exact binary representation... similar to how 1/3 has no exact decimal representation. So the arithmetic comes out very slightly wrong.

In [ ]:
print(0.1 + 0.2)
print(0.1 + 0.2 == 0.3)

The error shows up around the seventeenth digit. This can affect testing values for equality.
It can be useful to round to actually change the computed value. 

In [ ]:
print(np.round(0.1 + 0.2, 1))
print(np.round(0.1 + 0.2, 1) == 0.3)

**Note:** This isn't specific to Python... it's inherent to how computers store decimal numbers by default. Values are stored in binary (base 2), and decimal fractions like 0.1 can't be represented exactly in binary, only approximated. Excel has the same limitation but conceals it better, though not always consistently. Ask Claude to show you examples.

**Another note:** Be aware that Python's built-in round(), numpy, and pandas all round halves (0.5) toward the nearest even number rather than always up, so np.round(2.5) gives 2.0 while np.round(3.5) gives 4.0. This is intentional and standard in numerical computing because it avoids a systematic upward bias across many roundings, but it is not what most accounting conventions expect, so check it when a total has to tie.

**A fix for both problems:** avoid fractions altogether by working in the smallest unit. $100.10 becomes 10,010 cents... an integer, stored exactly, with no rounding convention to worry about until you convert back for display. Alternatively, consider `Decimal` (not taught here) when working with decimals.

### Identifying and handling missing values

Datasets often contain missing or unobserved values, and most analyses will either ignore them or report an error. It is important to understand why values are missing before handling them, and to be familiar with how the data source records them. Common conventions include:

* "NA", ".", or " " left in place of a value
* A code, like "99", that some data providers use to represent a missing value

Be cautious not to treat a code like 99 as a real number! In Python (`pandas`), missing values are referred to as `NaN` or `pd.NA`, and you can inspect them with `df[df['variable'].isna()]`. Our shipping data has one: the weight for invoice 1002 was blank in the source file. Let's find it in the long version of the data, where every value still sits in a single column.

In [ ]:
shipping_data_tidy[shipping_data_tidy['weight'].isna()]

Once you have found a missing value, you have a few options. One, you can obviously leave it blank. In `pandas`, many summary methods such as `mean()` and `std()` skip `NaN` values by default, while ordinary arithmetic involving `NaN` generally produces `NaN`. Alternatively, you can replace it with a set value (e.g., 0) using `df['variable'] = df['variable'].fillna(0)`, or you can drop the observation entirely with `df = df[df['variable'].notna()]`. For illustration purposes, let's replace the missing values with a 0.

In [ ]:
shipping_data_tidy['weight'] = shipping_data_tidy['weight'].fillna(0)

shipping_data_tidy

Here, filling with 0 is probably a bad idea. As always, context matters... in this case a shipment is unlikely to weigh 0. It is probably the case that the shipment simply wasn't weighed or the data was never entered. Below, we will convert the 0s for shipping weight back to missing.

In [ ]:
shipping_data_tidy['weight'] = shipping_data_tidy['weight'].replace(0, np.nan)

shipping_data_tidy

### Working with text

Like dates, text is also hard to work with (at least relative to numeric values). The `str` functions from `pandas` make simple text wrangling (relatively) easy. Here are some examples:

* `str.replace()` – replace certain characters from strings
* `str.lower()` – convert all characters to lower case (and vice versa with `str.upper()`)
* `str.strip()` – remove leading and trailing spaces

There are many other `str` functions in `pandas`! Turning to the invoices, remember that one of the customer names had a trailing space.

In [ ]:
invoices['customer_name'].unique()

One instance of `'Madison C. '` carries a trailing space. To Python that is a different string from `'Madison C.'`, so any filter or grouping on customer name would quietly treat them as two different customers. This is the kind of problem that is invisible and can cause problems downstream. Let's use `str.strip()` to remove leading and trailing spaces.

In [ ]:
invoices['customer_name'] = invoices['customer_name'].str.strip()

invoices['customer_name'].unique()

### Replacing values and conditionals

Sometimes, we may want to replace a value with another value, which we can do with `df['variable'] = df['variable'].replace({'oldvalue': 'new value'})`. Other times, we may want to assign values based on one or more conditions, which is what `.loc` is for:

```python
df.loc[condition1, 'variable'] = 'value1'
df.loc[condition2, 'variable'] = 'value2'
```

Let's use this to identify invoices as large or small based on a threshold. Here is where we are starting from, with the customer names now cleaned up.

In [ ]:
invoices

We assign every row the same value first, then overwrite the rows that meet our condition.

In [ ]:
invoices['invoice_size'] = 'Small' # Every invoice starts out small
invoices.loc[invoices['amount'] >= 200, 'invoice_size'] = 'Large'

invoices

### Identifying and handling duplicate values

"Duplicate" values are sometimes present in our data. It is important to understand why observations are duplicated before handling them. The `duplicated()` method flags every row that repeats an earlier row. Let's check whether any invoices are duplicated. Here, the invoice number should be unique (there shouldn't be two invoices with the same invoice number). Using the `subset` argument, we can list the variables that we want to use to check for duplicates (note the default is to search for duplicates in all columns).

In [ ]:
invoices.duplicated(subset='invoice')

Row 3 comes back as `True`. To see both copies rather than just the second one, pass the argument `keep=False` and use the result to filter the dataset.

In [ ]:
invoices[invoices.duplicated(subset='invoice', keep=False)]

Invoice 1002 was recorded twice and all other columns for the duplicated invoice are identical. In practice, we would still want to track down why this duplicate exists in the data. For now, let's assume this is truly a duplicate record and we can remove the second instance. We can do this using `drop_duplicates()`. Let's create a new invoice dataset with no duplicate invoices. The five rows above should become four.

In [ ]:
invoices_tidy = invoices.drop_duplicates(subset='invoice')

invoices_tidy

## Common data manipulation tools

With both datasets cleaned up, we can start asking questions of them using filter, select, sort, groupby, aggregate, and many more. You will get a lot of practice with `pandas` functions in DataCamp courses. Each adds a new tool to your arsenal!

### Filtering

Filtering extracts a "subset" of observations or rows in a dataset. Let's split the four invoices into Madison's invoices and everyone else's.

In [ ]:
madison_invoices = invoices_tidy[invoices_tidy['customer_name'] == 'Madison C.']

madison_invoices

In [ ]:
not_madison_invoices = invoices_tidy[invoices_tidy['customer_name'] != 'Madison C.']

not_madison_invoices

### Selecting

Selecting extracts variables or columns in a dataset. Here, we keep only the invoice and amount columns from Madison's invoices above.

In [ ]:
madison_invoices[['invoice', 'amount']]

### Sorting

`sort_values()` sorts data by different variables in ascending or descending order. Let's sort all the invoices by date from oldest to newest, and compare the order below to the `invoices` dataset above.

In [ ]:
invoices_tidy.sort_values(
    by='invoice_datetime', 
    ascending=True
)

Look at the index, the numbers running down the left side. Those are not row positions; they are labels `pandas` assigned when the data was first loaded, and they stay attached to their rows through everything we do. Sorting reordered the rows but carried each row's original label along with it, which is why the index reads 0, 1, 4, 2 rather than 0, 1, 2, 3. The missing 3 is left over from the duplicate row we dropped earlier.

Keeping the labels is often useful, since it lets you trace a row back to where it came from. `reset_index(drop=True)` discards the old labels and renumbers the rows 0, 1, 2, ... in their current order.

In [ ]:
invoices_tidy.sort_values(
    by='invoice_datetime',
    ascending=True
).reset_index(drop=True)

Same rows in the same order, now labeled 0 through 3. The `drop=True` argument is the part to remember: without it, `pandas` keeps the old labels by moving them into a new column named `index`, which is rarely what you want. Use `reset_index(drop=True)` when you want a fresh sequential index.

### Grouping and aggregating

`groupby()` lets us work with data in "groups" and `agg()` aggregates and summarizes the data. Let's group the invoices by customer and calculate the total amount for each customer. This collapses the four invoice rows into one row per customer, and it is where the trailing space we stripped earlier would have caused trouble. Notice the decimal issue that we noted above for Madison C. As above, we could fix that by rounding after aggregating.

In [ ]:
invoices_tidy.groupby('customer_name')['amount'].agg('sum')


## Combining data — vertically

Frequently, we have many files that contain segments of an entire dataset that we want to analyze. If the datasets *contain the same variables with different observations*, we might consider a vertical combination. For example, suppose that we have 4 quarterly sales datasets that we want to combine into one large annual dataset containing all sales for the year. `pd.concat()` is the function that allows us to stack or append datasets to one another (note this is the default in `pd.concat()`, we can change how datasets are appended using the `axis` argument). Importantly, these datasets have the same variables (columns) but different observations.

We already have two pieces to work with from the filtering section above: `madison_invoices` and `not_madison_invoices`. Stacking them should reconstruct the same set of rows, so this is an easy one to check.

In [ ]:
pd.concat(
    [madison_invoices, not_madison_invoices]
)

## Combining data — horizontally

In many cases, we may have several files that contain different information on observations. These datasets generally contain the same (or overlapping) observations with different variables. To combine these datasets, we need linking variables. Often, these are "unique identifiers" but they don't have to be.

Such combinations of data are often referred to as "merging" or "joining." There are several different kinds of "joins" that we can perform... best illustrated with Venn diagrams:

![Join Venn Diagram](joins_venn_diagram.png)

### Types of joins

In `pandas`, a join looks like `df_left.merge(df_right, how='left', on='key')`. The `on` argument specifies the identifier or key variable(s) you want to join on, and `how` is the type of join you want to perform.

| `how` | What it keeps |
|-------|---------------|
| `'left'` | All rows in the left (L) dataset |
| `'right'` | All rows in the right (R) dataset (the reverse of a left join) |
| `'inner'` | Only the rows that match in both |
| `'outer'` | All rows from both datasets |

Note that if both datasets have non-key columns with the same name, they will be given a suffix to indicate which dataset the variable comes from (`_x` represents left and `_y` represents right).

Our two cleaned datasets share an `invoice` column, so that is our key. Before merging anything, look at both sides and note how many rows each one has and which invoice numbers appear in each.

In [ ]:
invoices_tidy

In [ ]:
shipping_data_tidy

Four invoices, three shipments. Invoice 1004 never shows up in the shipping data, so the join has to decide what to do with it. A left join keeps every row on the left and leaves the shipping columns empty where there is no match.

In [ ]:
invoice_ship_left = invoices_tidy.merge(
    shipping_data_tidy,
    on='invoice',
    how='left'
)

invoice_ship_left

All four invoices from the left dataset survived, and invoice 1004 came through with `NaN` in every shipping column. An inner join keeps only the matches, so the same merge with `how='inner'` should drop that invoice entirely and leave us with three rows.

In [ ]:
invoice_ship_inner = invoices_tidy.merge(
    shipping_data_tidy,
    on='invoice',
    how='inner'
)

invoice_ship_inner

### Filtering joins

An "anti-join" retains rows in the left dataset that do not match to the right dataset. This can be useful when we want a list of rows that are missing from another dataset, which in an audit setting might be exactly the list you care about. The pattern is a left join with `indicator=True`.

In [ ]:
invoice_shipment_check = invoices_tidy.merge(
    shipping_data_tidy[['invoice']],
    on='invoice',
    how='left',
    indicator=True # Adds a _merge column showing where each row came from
)

invoice_shipment_check

The `_merge` column tells us where each row came from. `both` means the invoice matched a shipment, and `left_only` means it did not. Filtering on that column leaves us with the invoices we actually want to look into.

In [ ]:
no_shipment = invoice_shipment_check[
    invoice_shipment_check['_merge'] == 'left_only'
]

no_shipment

### Join cardinality

Beyond the type of join, it matters how many rows on each side can match a given key. There are three cases.

In a **one-to-one** join, each key in the left DataFrame can only match to one key in the right DataFrame. It combines rows from both DataFrames based on a common key or index, and because no duplicate keys are allowed in either DataFrame, it preserves one row per key in the result. Here, you will never have instances where the merged dataset has duplicated keys. Because each key appears at most once on each side, matching a key cannot multiply rows. For left, right, and inner joins, the merge therefore cannot create additional rows through duplicated matches. This is what you want when merging related datasets with unique identifiers.

In a **one-to-many** join, each key in the left DataFrame can match multiple rows in the right DataFrame. The left DataFrame must have unique keys, while the right can have duplicates. It combines rows by repeating the left-side row for each matching right-side row, so the resulting table can have more rows than the left DataFrame. This is useful when joining multiple related records onto unique identifiers, such as users and transactions, and it allows us to merge richer, more granular information onto those identifiers. Note that **many_to_one** is the reverse orientation of **one-to-many**.

In a **many-to-many** join, keys in both DataFrames can appear multiple times, and each matching combination of keys results in a new row in the output. This can produce a "Cartesian product" for matching key pairs, so the result may have *many* more rows than either input DataFrame. It is used when both datasets contain repeated entries for the same key, and should be used carefully to avoid unexpectedly large join results.

The `validate` argument in `merge()` lets you check which one you actually have. Pass `'one_to_one'` or `'one_to_many'` (`'many_to_one'`) and `pandas` will raise an error if the data does not match what you expected.

Below, we join the shipping data back to the original invoices data that contained duplicates. The left side has three unique shipments, but the right side still contains invoice 1002 twice, so this is a genuine one-to-many join. Watch the three rows on the left become four in the result.

In [ ]:
shipping_data_tidy.merge(
    invoices,
    on='invoice',
    how='left',
    validate='one_to_many' # Errors out if the left keys are not unique
)

If we wanted to impose that the same match should be `'one_to_one'`, then we would get an error.

**Caution:** Always check your joins. Make sure the number of observations before and after a join make sense! If something seems off, inspect for duplicates and check the join keys.

## Sampling the data

As accountants we often select "samples" of our data. There are many reasons to do this, and `pandas` has several useful tools. 

Without fixing the random seed, a random sample will generally differ each time the code is run. We can make the result reproducible by setting `random_state`. 

To take a random sample of a fixed number of observations, use `sample(n=3)`.

In [ ]:
invoices_tidy.sample(n=3, random_state=42)

To take a share of the observations instead of a fixed count, use `sample(frac=0.5)` for 50 percent.

In [ ]:
invoices_tidy.sample(frac=0.5, random_state=42)

You can also take a random sample weighted by a variable, using the `weights` argument, so that observations with larger values are more likely to be selected. Weighting by amount here means the larger invoices are more likely to end up in the sample, which is a common approach in audit sampling.

In [ ]:
invoices_tidy.sample(n=2, weights='amount', random_state=42)

In many settings, we sample without replacement where we only allow each datapoint to be sampled once. Sampling with replacement allows observations to be sampled more than once (i.e., each observation in a sample drawn from the entire population, such that the same observation could be selected more than once). The default in `sample()` is without replacement. It can be changed with the option `replace=True`.

In [ ]:
invoices_tidy.sample(n=3, random_state=123, replace=True)